In [ ]:
Investigation found WIP session_status_src_name logic was incomplete. Future booked sessions were returning NULL/not being added, and cancellation logic was not aligned to the definition. This caused downstream session_status_id mapping to be blank for those records.

In [ ]:
SELECT
    COALESCE(
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
        END,
        'NULL'
    ) AS old_logic_output,

    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(COALESCE(st.description, '')) LIKE '%cancel%'
            THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time > current_timestamp()
            THEN 'Booked'
        WHEN e.activity_date_time < current_timestamp()
            THEN 'Attended'
        ELSE 'Unknown'
    END AS expected_logic_output,

    COUNT(*) AS row_count
FROM silver_wip_activityentry e
LEFT JOIN silver_wip_activitytype st
    ON e.activity_type_id = st.id
GROUP BY
    COALESCE(
        CASE
            WHEN e.is_dna = true THEN 'Did Not Attend'
            WHEN e.activity_date_time < current_timestamp() THEN 'Attended'
        END,
        'NULL'
    ),
    CASE
        WHEN e.is_dna = true THEN 'Did Not Attend'
        WHEN LOWER(COALESCE(st.description, '')) LIKE '%cancel%'
            THEN 'Cancelled with greater than 24 hours notice'
        WHEN e.activity_date_time > current_timestamp()
            THEN 'Booked'
        WHEN e.activity_date_time < current_timestamp()
            THEN 'Attended'
        ELSE 'Unknown'
    END
ORDER BY row_count DESC;